# Neural ODEs (NODEs) for Ballistic Motion
In this demo we will use a NODE to capture *residual* dynamics in a system of ODEs. You can use NODEs directly, but there are many demonstrations of this online, and this is a better way to use NODEs since it allows you to incorporate known physics alongside the NN.

In [ ]:
import torch
import utils

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch.nn as nn
import torch.utils.data as data

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

from pytorch_lightning import seed_everything
from torchdiffeq import odeint as td_odeint

In [ ]:
%load_ext autoreload
%autoreload 2

# 1. Fit the Dynamics with a NN

## Generate Training Data

In [ ]:
# Generate a ballistic trajectory in (x, y) to use as training data.
# Here we will use 3 balls of different masses thrown from different places with different initial conditions

In [ ]:
from scipy.integrate import odeint as sp_odeint

def ballistic(state, t, m=1.0):
    # this system is actually autonomous so does not depend on t
    x, y, vx, vy = state
    dsdt = [vx, vy, 0.0, -9.8]
    return dsdt

trajectories = []
for i, (m, s0) in enumerate(
    zip(
        [1.0, 2.0, 3.0], 
        [
            [0.0, 0.0, 10.0, 10.0], # x0, y0, vx0, vy0
            [2.0, 0.0, 5.0, 10.0], 
            [4.0, 0.0, 10.0, 5.0]
        ]
    )
):
    t = np.linspace(0, 2, 101)
    traj = sp_odeint(ballistic, s0, t, args=(m,))
    trajectories.append(
        utils.Trajectory2D(t=t, x=traj[:, 0], y=traj[:, 1], vx=traj[:, 2], vy=traj[:, 3], mass=m)
    )

In [ ]:
xm = 0
xM = 25
ym = -10
yM = 8

fig = go.Figure(
    data=[
        go.Scatter(x=[p[1] for p in traj_], y=[p[2] for p in traj_],
                     mode="lines",
                     line=dict(width=2, color=color)) for color, traj_ in zip(['blue', 'green', 'orange'], trajectories)
    ] + 
    [
        go.Scatter(x=[traj_[0][1]], y=[traj_[0][2]],
                     mode="markers",
                     marker=dict(color=color, size=10)) for color, traj_ in zip(['blue', 'green', 'orange'], trajectories)
    ]
)
fig.update_layout(
    width=600, 
    height=450,
    xaxis=dict(range=[xm, xM], autorange=False, zeroline=False),
    yaxis=dict(range=[ym, yM], autorange=False, zeroline=False),
    title_text="Kinematics", 
    title_x=0.5,
    updatemenus = [
        dict(
            type = "buttons",
            buttons = [
                dict(
                    args = [None, {"frame": {"duration": 20, "redraw": False}, "fromcurrent": True, "transition": {"duration": 20}}],
                    label = "Play",
                    method = "animate",
                    )
            ]
        )
    ]
)

fig.update(frames=[
    go.Frame(
       data=[go.Scatter(x=[traj_[k][1]], y=[traj_[k][2]]) for traj_ in trajectories],
       traces=[3+i for i in range(len(trajectories))]
   ) for k in range(len(trajectories[0]))
])

fig.show()

In [ ]:
# Create sets of t_start and t_end points for various dt, for all trajectories
X_train, y_train = utils.build_time_lagged_set(trajectories, max_incrs=3)

In [ ]:
X_train[:6]

In [ ]:
y_train[:6]

In [ ]:
len(X_train), len(y_train)

## Train a NODE

In [ ]:
torch.manual_seed(0)
# torch.set_default_dtype(dtype)
# device = torch.device(device) 

In [ ]:
seed_everything(42, workers=True)

In [ ]:
def build_loaders(
    X_train,
    y_train,
    batch_size: int = 20, 
    train_frac: float = 0.8, 
    device: str = 'cpu', 
    dtype: torch.dtype = torch.float32, 
    shuffle: bool = False, 
    drop_last: bool = True,
    seed: int = 42
) -> torch.utils.data.dataloader.DataLoader:
    train = data.TensorDataset(
        torch.tensor(X_train, dtype=dtype, requires_grad=True).to(device), 
        torch.tensor(y_train, dtype=dtype, requires_grad=True).to(device)
    )
    train_set_size = int(len(train) * train_frac)
    valid_set_size = len(train) - train_set_size

    seed = torch.Generator().manual_seed(seed)
    train_set, valid_set = data.random_split(train, [train_set_size, valid_set_size], generator=seed)

    train_loader = data.DataLoader(
        train_set, 
        batch_size=batch_size, 
        shuffle=shuffle, # Also possible because X and y are matched correctly
        drop_last=drop_last, # To avoid "spikes" from uneven dataset sizes
    )

    valid_loader = data.DataLoader(
        valid_set, 
        batch_size=batch_size, 
        shuffle=False, # Unnecessary for validation set
        drop_last=False, # No need to drop from validation set
    )

    return train_loader, valid_loader

In [ ]:
train_loader, valid_loader = build_loaders(X_train, y_train, batch_size=20, train_frac=0.8, device='cuda')

In [ ]:
class BallisticDynamics(utils.Dynamics): 
    def forward(self, t: torch.Tensor, state: torch.Tensor) -> torch.Tensor:
        """
        Compute the derivative of the state with respect to time.

        This example computes certain aspects analytically with a closed form, then uses a NN to compute a residual.

        Parameters
        ----------
        t : torch.tensor(ndim=0)
            Current time of the system's state. Ignored for autonomous systems.

        state : torch.tensor(ndim=1)
            Current system state, e.g., [x, y, x', y'].

        Returns
        -------
        deriv : torch.tensor(ndim=1)
            Derivative of system state, e.g., [x', v', x'', y''].
        """
        vx = state[2:3]
        vy = state[3:4]

        ax = torch.zeros_like(vx)
        ay = torch.zeros_like(vy)

        # "Known" part of the dynamics
        ax[:] = 0.0
        ay[:] = -8.0 # y'' = -8.0, missing the -1.8 amount = -9.8

        # "Unknown" part of the dynamics, can make dependent upon select variables
        resid = self.vector_field_resid(state) 

        # Combine to make the net dynamics
        ax += resid[0:1] # Should target 0 always
        ay += resid[1:2] # Should target -1.8 always

        return torch.cat([vx, vy, ax, ay], axis=-1)

In [ ]:
vector_field_resid = nn.Sequential(
        nn.Linear(2*2, 64), # [x, y, vx, vy] - autonomous
        nn.Tanh(), 
        nn.Linear(64, 2) # [x'', y'']
)
system = BallisticDynamics(vector_field_resid=vector_field_resid)

learner = utils.Learner(ndim=2, system=system)

trainer = pl.Trainer(
    min_epochs=1, 
    max_epochs=2, 
    accelerator="gpu",
    devices="auto",
    precision=64,
    default_root_dir="checkpoints/",
    callbacks=[pl.callbacks.early_stopping.EarlyStopping(monitor="val_loss", min_delta=0.00, patience=5, verbose=False, mode="min")],
    log_every_n_steps=1,
    enable_progress_bar=True,
    enable_checkpointing=True,
    deterministic=True,
    inference_mode=True,
    profiler=None
)

trainer.fit(
    model=learner, 
    train_dataloaders=train_loader, 
    val_dataloaders=valid_loader
)

# tensorboard --logdir .

In [ ]:
# At minimum, train for a while and monitor loss and predictive performance (plot)

# Future optimizations
# Callbacks for pl
# how to save over time and report
# make device and dtype consistent
# learning rate updates
# profiling for bottlenecks -> num_workers in DataLoader

In [ ]:
# Check that the dynamics model is outputting ~ [0, -1.8] for all inputs
m_ = learner.system.vector_field_resid
m_.eval()
with torch.no_grad():
    pred_ = m_(torch.tensor(X_train[:, 1:-1])) # [x, y, vx, vy] is input for this model

torch.mean(pred_, axis=0) # Very close to target of [0, -1.8]!

In [ ]:
plt.plot(pred_[:, 0])
plt.plot(pred_[:, 1])
plt.axvline(891/3, color='r', alpha=0.5)
plt.axvline(891/3*2, color='r', alpha=0.5)
plt.axhline(0.0, color='k', alpha=0.5)
plt.axhline(-1.8, color='k', alpha=0.5)

In [ ]:
preds = trainer.predict(dataloaders=valid_loader)

In [ ]:
traj_ = torch.vstack(preds)
plt.plot(traj_[:, 0], traj_[:, 1], 'o', label='Predictions (Validation Set)')

cutoffs = np.array([0, 1, 2, 3], dtype=int)*len(X_train)//3
for i in range(3):
    plt.plot(X_train[cutoffs[i]:cutoffs[i+1], 1], X_train[cutoffs[i]:cutoffs[i+1], 2], '-', label=f'Ball {i+1} Observed Path')
plt.legend(loc='best')
plt.xlabel('x')
plt.ylabel('y')

# 2.  Use PySR to Represent the NN in Closed Form (see `demo/symbolic_regression`)

In [ ]:
# Here, the NN in the NODE accepts the state [x, y, vx, vy] as input so use this as training input; the output is the NN output since we are trying to mimic the 
# network. PySR seems to be setup to only regress scalar outputs so we will need to train a separate model for x'' and y'', the outputs of the NN.

# You can fit PySR with noise
# https://astroautomata.com/PySR/examples/

In [ ]:
def write_data(X, y, filename):
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1, 1)
    data = np.concatenate((X, y), axis=1)
    np.savetxt(filename, data)        

In [ ]:
# # Write the x'' and y'' outputs of the trained NN for PySR
# write_data(X_train[:, 1:-1], pred_[:, 0], filename='ax.txt')
# write_data(X_train[:, 1:-1], pred_[:, 1], filename='ay.txt')

Generally best not to do this in Jupyter - run from command line like (see `demo/symbolic_regression` for details)

~~~python
$ conda activate project-env
$ python pysr_demo.py
~~~

In [ ]:
# Observe this works pretty well!

# Need to validate: can rank based on complexity or use expressions to predict a held-out validation set and use this error to select
# Should balance (1) speed to evaluate and (2) validation accuracy